### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_customer_satisfaction",
    dataset_year="2016",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/santander-customer-satisfaction",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-customer-satisfaction -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_customer_satisfaction && mv train.csv local-data-warehouse/santander_customer_satisfaction/
""",
    # References
    academic_reference_bibtex=r"""@misc{Jimenez2016SantanderCustomerSatisfaction,
  author = {Soraya Jimenez and Will Cukierski},
  title  = {Santander Customer Satisfaction},
  year   = {2016},
  howpublished = {\url{https://kaggle.com/competitions/santander-customer-satisfaction}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Jimenez2016SantanderCustomerSatisfaction",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- We remove duplicated columns and zero-variance columns.
- We add a zero-count feature (number of zeros per row). #1 Kaggle experts also applied a lot of other features and feature selection but each experts used something different (https://www.kaggle.com/competitions/santander-customer-satisfaction/writeups/1-leustagos-3rd-place-solution).
- We drop duplicates as their origin is unclear and they could create data leaks when splitting the training data.
- We replace -999999 in var3 with with nan.
- We keep the data otherwise as it. Several columns might be categorical and contain encoded nan values but we are not able to determine this due to anonymization.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="TARGET",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="TARGET",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# remove duplicated columns
df = df.loc[:, ~df.T.duplicated()]
# remove zero-variance columns
df = df.loc[:, df.nunique(dropna=False) > 1]
# add zero-count feature
numeric_cols = df.drop(columns=["ID", "TARGET"])
df["zero_count"] = (numeric_cols == 0).sum(axis=1)
df = df.drop(columns=["ID"])
df[task_mold.target_column_name] = df[task_mold.target_column_name].astype("category")

# Drop duplicates w/o target col
df = df.drop_duplicates(subset=df.columns.difference([task_mold.target_column_name]))

# Replace -999999 with np.nan
df["var3"] = df["var3"].replace(-999999, np.nan)

df = df.reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()